# Simulation timing profile

This notebook profiles the sample-data simulation loop with optional warmup and repeated runs.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from config import get_development_config, setup_dev_directories, get_simulation_params
from src.caching import load_simulation_caches
from src.data_loader import load_electricity_assets, load_hazard_maps
from src.simulation import simulate_asset_damage_recovery_access_breakdown
from src.timing_profiler import (
    SimulationTimingProfiler,
    run_profiled_runs,
    summarize_profiled_runs,
)


In [ ]:
config = get_development_config()
setup_dev_directories(config, remove_cache=False)
simulation_params = get_simulation_params(config)
config['simulation_config']['major_timestep'] = 24

hazard_maps = load_hazard_maps(config['hazard_dir'], max_days=None)
gdf_assets = load_electricity_assets(config['electricity_dir'], asset_types=['msls'])
caches = load_simulation_caches(config['interim_dir'], config['hazard_dir'])

print(f'Assets: {len(gdf_assets)}')
print(f'Hazard maps: {len(hazard_maps)}')

In [ ]:
def run_profiled_sample(profiler, *, execution_id='timing_profile_run'):
    return simulate_asset_damage_recovery_access_breakdown(
        gdf_assets,
        hazard_maps,
        number_repair_crews=simulation_params['number_repair_crews'],
        repair_crew_assignment_method=simulation_params['repair_crew_assignment_method'],
        flood_threshold=simulation_params['flood_threshold'],
        recovery_parameters=simulation_params['recovery_parameters'],
        root_dir=config['root_dir'],
        verbose=False,
        timestep_output=True,
        execution_id=execution_id,
        config=config,
        major_timestep=config['simulation_config']['major_timestep'],
        accessibility_cache=caches.get('accessibility_cache'),
        hazard_extraction_cache=caches.get('hazard_extraction_cache'),
        overlap_cache=caches.get('overlap_cache'),
        island_cache=caches.get('island_cache'),
        profiler=profiler,
    )


In [ ]:
# Warmup + single profiled run
profiled_runs = run_profiled_runs(
    lambda profiler: run_profiled_sample(profiler, execution_id=f'timing_{int(time.time())}'),
    runs=1,
    warmup_runs=1,
)

single_profiler = profiled_runs[0]['profiler']
phase_summary = single_profiler.phase_summary()
timestep_summary = single_profiler.timestep_summary()
timestep_phase_summary = single_profiler.timestep_phase_summary()

print('=== Overall phase totals ===')
print(phase_summary.to_string(index=False))
print('\n=== Per-phase call count / average ===')
print(phase_summary[['phase', 'call_count', 'avg_seconds']].to_string(index=False))
print('\n=== Per-timestep timing summary ===')
print(timestep_summary[['timestep', 'total_seconds']].to_string(index=False))
print('\n=== Per-timestep phase aggregate ===')
print(timestep_phase_summary.to_string(index=False))

In [ ]:
# EMA-workbench style repeated timing runs
from ema_workbench import Model, IntegerParameter, Constant, ScalarOutcome, SequentialEvaluator

def ema_profile_run(run_id=0):
    profiler = SimulationTimingProfiler()
    run_profiled_sample(profiler, execution_id=f'ema_timing_{run_id}_{int(time.time())}')

    phase_df = profiler.phase_summary(sort_desc=False)
    timestep_df = profiler.timestep_summary()

    sim_total = float(phase_df.loc[phase_df['phase'] == 'simulation.total', 'total_seconds'].sum())
    dep_total = float(phase_df.loc[phase_df['phase'].str.startswith('dependency.'), 'total_seconds'].sum())
    timestep_avg = float(timestep_df['total_seconds'].mean()) if not timestep_df.empty else 0.0

    return {
        'simulation_total_seconds': sim_total,
        'dependency_total_seconds': dep_total,
        'timestep_avg_seconds': timestep_avg,
    }

timing_model = Model('TimingSimulation', function=ema_profile_run)
timing_model.uncertainties = [IntegerParameter('run_id', 0, 4)]
timing_model.constants = [Constant('run_id', 0)]
timing_model.outcomes = [
    ScalarOutcome('simulation_total_seconds'),
    ScalarOutcome('dependency_total_seconds'),
    ScalarOutcome('timestep_avg_seconds'),
]

with SequentialEvaluator(timing_model) as evaluator:
    experiments, outcomes = evaluator.perform_experiments(scenarios=5)

ema_runs_df = pd.DataFrame({
    'run_index': range(len(outcomes['simulation_total_seconds'])),
    'simulation_total_seconds': outcomes['simulation_total_seconds'],
    'dependency_total_seconds': outcomes['dependency_total_seconds'],
    'timestep_avg_seconds': outcomes['timestep_avg_seconds'],
})
print('=== Repeated-run aggregate summary (EMA) ===')
print(ema_runs_df.describe().T[['mean', 'min', 'max']].to_string())

In [ ]:
# Optional aggregate from native helper over profiled runs
multi_runs = run_profiled_runs(
    lambda profiler: run_profiled_sample(profiler, execution_id=f'batch_{int(time.time())}'),
    runs=3,
    warmup_runs=1,
)
print(summarize_profiled_runs(multi_runs).to_string(index=False))